# Day 10: End-to-End Pipeline Walkthrough
This notebook ties together the entire Medallion journey for a synthetic record.

In [ ]:
import duckdb
import hashlib
from datetime import datetime
import json

con = duckdb.connect(':memory:')

# 1. RAW INGESTION (Mock HL7 JSON)
raw_hl7 = {
    "resourceType": "Patient",
    "id": "MRN-999",
    "name": [{"family": "Doe", "given": ["Jane"]}],
    "gender": "female",
    "birthDate": "1985-10-25"
}
print("1. RAW PAYLOAD:", json.dumps(raw_hl7))

# 2. BRONZE LAYER (Appended)
con.execute("CREATE TABLE bronze (payload_id VARCHAR, json_data VARCHAR, ingestion_time TIMESTAMP)")
con.execute("INSERT INTO bronze VALUES ('payload-abc', ?, current_timestamp())", [json.dumps(raw_hl7)])
print("\n2. BRONZE LAYER:")
print(con.execute("SELECT * FROM bronze").df())

# 3. DE-IDENTIFICATION & SILVER OMOP
def mask_mrn(mrn):
    return "FPE-" + hashlib.sha256(mrn.encode()).hexdigest()[:8]

parsed = json.loads(con.execute("SELECT json_data FROM bronze").fetchone()[0])
masked_mrn = mask_mrn(parsed['id'])
gender_concept = 8532 if parsed['gender'] == 'female' else 8507

con.execute("CREATE TABLE silver_omop_person (person_id BIGINT, gender_concept_id INT, person_source_value VARCHAR)")
con.execute(f"INSERT INTO silver_omop_person VALUES (999, {gender_concept}, '{masked_mrn}')")
print("\n3. SILVER OMOP LAYER (De-Identified):")
print(con.execute("SELECT * FROM silver_omop_person").df())

# 4. DATA QUALITY (Circuit Breaker)
# Rule: gender_concept_id must be valid OMOP concept (8507 or 8532)
dq_passed = con.execute("SELECT COUNT(*) FROM silver_omop_person WHERE gender_concept_id IN (8507, 8532)").fetchone()[0] == 1
print(f"\n4. DATA QUALITY PASSED: {dq_passed}")

# 5. GOLD LAYER (Mock dbt agg)
con.execute("CREATE VIEW gold_demographics AS SELECT gender_concept_id, COUNT(*) as count FROM silver_omop_person GROUP BY gender_concept_id")
print("\n5. GOLD LAYER (Demographic Rollup):")
print(con.execute("SELECT * FROM gold_demographics").df())
